# Challenge 6.4 - ReadyNow Final

Root coordinator -> search research draft -> critique -> refine. Weather and route specialists provide emergency-preparedness support; callbacks validate and log requests.

In [ ]:
%pip install -q --upgrade google-adk
import os
from datetime import datetime,timezone
import vertexai
from google.adk.agents import LlmAgent,SequentialAgent
from google.adk.tools import google_search,agent_tool
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest,LlmResponse
from google.genai import types
from vertexai.preview import reasoning_engines
PROJECT_ID=os.environ.get('GOOGLE_CLOUD_PROJECT','qwiklabs-gcp-02-9e12deb8c42f'); MODEL='gemini-2.5-flash'; vertexai.init(project=PROJECT_ID,location='us-central1'); AUDIT_LOG=[]
def validate_and_log(callback_context:CallbackContext,llm_request:LlmRequest):
    text=' '.join(p.text or '' for c in llm_request.contents or [] if c.role=='user' for p in c.parts or []); AUDIT_LOG.append({'time':datetime.now(timezone.utc).isoformat(),'input':text})
    if any(x in text.lower() for x in ('ignore previous','system prompt','jailbreak','api key')): return LlmResponse(content=types.Content(role='model',parts=[types.Part(text='I only assist with emergency preparedness and safety information.')]))
def log_response(callback_context,llm_response): AUDIT_LOG.append({'time':datetime.now(timezone.utc).isoformat(),'stage':'response'})


In [ ]:
search=LlmAgent(name='search',model=MODEL,instruction='Research official emergency information and write a factual draft.',tools=[google_search],output_key='research_draft')
weather=LlmAgent(name='weather',model=MODEL,instruction='Provide U.S. weather preparedness guidance.')
routes=LlmAgent(name='routes',model=MODEL,instruction='Give route preparedness guidance; follow official evacuation orders.')
critique=LlmAgent(name='critique',model=MODEL,instruction='Review {research_draft} for safety and accuracy.',output_key='critique_notes')
refine=LlmAgent(name='refine',model=MODEL,instruction='Write the final answer from {research_draft} and {critique_notes}.',output_key='final_answer')
flow=SequentialAgent(name='research_critique_refine',sub_agents=[search,critique,refine])
root_agent=LlmAgent(name='readynow_root',model=MODEL,instruction='Coordinate emergency preparedness. Always send requests through research_critique_refine; use weather and routes when needed.',sub_agents=[weather,routes,flow],before_model_callback=validate_and_log,after_model_callback=log_response)
app=reasoning_engines.AdkApp(agent=root_agent); print('ReadyNow 6.4 created.')


In [ ]:
s=app.create_session(user_id='tester'); sid=s['id'] if isinstance(s,dict) else s.id
for e in app.stream_query(user_id='tester',session_id=sid,message='Give hurricane preparedness advice for a family in Miami, Florida.'):
    print(e)
print('Audit log entries:',len(AUDIT_LOG))